# Rainbow-lite DQN — CartPole-v1

All-in-one notebook combining Network, Replay Buffer, Agent, and Training Loop. Configured for a quick ~5 minute run.

In [ ]:
# Install dependencies if needed
# !pip install gymnasium numpy matplotlib

In [ ]:
"""
network.py  —  Dueling Q-Network (NumPy)
=========================================
Architecture
------------
Input (4) ──► Shared MLP  ──► Value stream  V(s):  scalar
                           └──► Advantage stream A(s,a): 2-dim

Q(s,a) = V(s) + [ A(s,a) - mean_a(A(s,a)) ]

Why Dueling?
  • The agent can learn *which states are valuable* (V) without having to
    learn the effect of each action (A) in every state.
  • Leads to faster, more stable convergence — especially powerful when
    many actions have similar expected value (e.g. left≈right while pole
    is nearly vertical).

Optimizer: Adam with bias-correction.
Loss:      Huber (smooth-L1), robust to large TD-error outliers.
           δ = 1.0  ⟹  L1 for |err|>1, L2 for |err|≤1.
Gradient:  Global-norm clipping at 10.0.
"""

from __future__ import annotations

import numpy as np

# ---------------------------------------------------------------------------
# Helpers
# ---------------------------------------------------------------------------


def _relu(x: np.ndarray) -> np.ndarray:
    return np.maximum(0.0, x)


def _relu_grad(x: np.ndarray) -> np.ndarray:
    return (x > 0.0).astype(np.float32)


def _huber(errors: np.ndarray, delta: float = 1.0) -> tuple[np.ndarray, np.ndarray]:
    """
    Huber loss and its gradient w.r.t. errors.

    Returns
    -------
    loss_per_sample : (batch,)
    grad_per_sample : (batch,)  — d(Huber)/d(error)
    """
    abs_err = np.abs(errors)
    quadratic = np.minimum(abs_err, delta)
    loss = 0.5 * quadratic**2 + delta * (abs_err - quadratic)
    grad = np.where(abs_err <= delta, errors, delta * np.sign(errors))
    return loss, grad


# ---------------------------------------------------------------------------
# Parameter block helper
# ---------------------------------------------------------------------------


class _Params:
    """Holds one weight matrix + bias + Adam moments."""

    def __init__(self, shape_in: int, shape_out: int, gain: float = 2.0) -> None:
        scale = np.sqrt(gain / shape_in)
        self.W = np.random.randn(shape_in, shape_out).astype(np.float32) * scale
        self.b = np.zeros(shape_out, dtype=np.float32)
        self.mW = np.zeros_like(self.W)
        self.vW = np.zeros_like(self.W)
        self.mb = np.zeros_like(self.b)
        self.vb = np.zeros_like(self.b)

    def update(
        self,
        dW: np.ndarray,
        db: np.ndarray,
        lr: float,
        beta1: float,
        beta2: float,
        eps: float,
        t: int,
    ) -> None:
        """One Adam step (in-place)."""
        bc1 = 1.0 - beta1**t
        bc2 = 1.0 - beta2**t
        self.mW = beta1 * self.mW + (1 - beta1) * dW
        self.vW = beta2 * self.vW + (1 - beta2) * dW**2
        self.W -= lr * (self.mW / bc1) / (np.sqrt(self.vW / bc2) + eps)
        self.mb = beta1 * self.mb + (1 - beta1) * db
        self.vb = beta2 * self.vb + (1 - beta2) * db**2
        self.b -= lr * (self.mb / bc1) / (np.sqrt(self.vb / bc2) + eps)

    def clone(self) -> _Params:
        p = _Params.__new__(_Params)
        p.W = self.W.copy()
        p.b = self.b.copy()
        p.mW = self.mW.copy()
        p.vW = self.vW.copy()
        p.mb = self.mb.copy()
        p.vb = self.vb.copy()
        return p


# ---------------------------------------------------------------------------
# Dueling Q-Network
# ---------------------------------------------------------------------------


class DuelingQNetwork:
    """
    Dueling Double-DQN Q-network.

    Parameters
    ----------
    state_dim  : int   – observation size (4 for CartPole)
    action_dim : int   – number of actions (2 for CartPole)
    hidden     : int   – neurons per shared hidden layer
    lr         : float – Adam learning rate
    huber_delta: float – Huber loss threshold
    clip_norm  : float – global gradient-norm clip value
    """

    BETA1 = 0.9
    BETA2 = 0.999
    EPS_ADAM = 1e-8

    def __init__(
        self,
        state_dim: int = 4,
        action_dim: int = 2,
        hidden: int = 128,
        lr: float = 5e-4,
        huber_delta: float = 1.0,
        clip_norm: float = 10.0,
    ) -> None:
        self.action_dim = action_dim
        self.lr = lr
        self.huber_delta = huber_delta
        self.clip_norm = clip_norm
        self._t = 0  # Adam step counter

        # Shared trunk (2 hidden layers)
        self.fc1 = _Params(state_dim, hidden)
        self.fc2 = _Params(hidden, hidden)

        # Value stream:     hidden → 64 → 1
        self.v1 = _Params(hidden, 64)
        self.v2 = _Params(64, 1)

        # Advantage stream: hidden → 64 → action_dim
        self.a1 = _Params(hidden, 64)
        self.a2 = _Params(64, action_dim)

        # Cached forward activations (set during forward pass)
        self._cache: dict = {}

    # ------------------------------------------------------------------
    # Forward pass
    # ------------------------------------------------------------------

    def forward(self, s: np.ndarray) -> np.ndarray:
        """
        s : (batch, 4)  →  Q-values : (batch, 2)
        """
        # --- Shared trunk ---
        z1 = s @ self.fc1.W + self.fc1.b
        a1 = _relu(z1)
        z2 = a1 @ self.fc2.W + self.fc2.b
        a2 = _relu(z2)

        # --- Value stream ---
        zv1 = a2 @ self.v1.W + self.v1.b
        av1 = _relu(zv1)
        V = av1 @ self.v2.W + self.v2.b  # (batch, 1)

        # --- Advantage stream ---
        za1 = a2 @ self.a1.W + self.a1.b
        aa1 = _relu(za1)
        A = aa1 @ self.a2.W + self.a2.b  # (batch, action_dim)

        # --- Dueling aggregation ---
        Q = V + (A - A.mean(axis=1, keepdims=True))

        # Cache for backprop
        self._cache = dict(
            s=s,
            z1=z1,
            a1=a1,
            z2=z2,
            a2=a2,
            zv1=zv1,
            av1=av1,
            V=V,
            za1=za1,
            aa1=aa1,
            A=A,
            Q=Q,
        )
        return Q

    # ------------------------------------------------------------------
    # Backward pass — IS-weighted Huber loss
    # ------------------------------------------------------------------

    def backward(
        self,
        s: np.ndarray,
        targets: np.ndarray,
        actions: np.ndarray,
        weights: np.ndarray,  # IS weights from PER, shape (batch,)
    ) -> tuple[float, np.ndarray]:
        """
        Weighted Huber loss over chosen actions only.

        Returns
        -------
        loss    : float        – scalar mean loss (for logging)
        td_errs : (batch,)     – raw TD errors (for PER priority update)
        """
        batch = s.shape[0]
        Q = self.forward(s)

        td_errs = Q[np.arange(batch), actions] - targets  # (batch,)
        huber_loss, huber_grad = _huber(td_errs, self.huber_delta)
        loss = float(np.mean(weights * huber_loss))

        # Gradient into Q-values:  only the taken action column is non-zero
        dQ = np.zeros_like(Q)
        dQ[np.arange(batch), actions] = weights * huber_grad / batch

        # --- Dueling aggregation backward ---
        # Q = V + A - mean(A)  ⟹  dV = sum(dQ), dA = dQ - mean(dQ)
        dV = dQ.sum(axis=1, keepdims=True)  # (batch, 1)
        dA = dQ - dQ.mean(axis=1, keepdims=True)  # (batch, action_dim)

        # --- Advantage stream backward (a1: hidden→64, a2: 64→action_dim) ---
        # Layer a2: (batch, 64) @ (64, action_dim)  → dA is (batch, action_dim)
        daa1 = dA @ self.a2.W.T  # (batch, 64)
        dza1 = daa1 * _relu_grad(self._cache["aa1"])  # (batch, 64)
        dA1_W = self._cache["aa1"].T @ dA  # (64, action_dim)
        dA1_b = dA.sum(0)  # (action_dim,)
        # Propagate through a1 (hidden → 64): W shape (hidden, 64)
        dA0_W = self._cache["a2"].T @ dza1  # (hidden, 64)
        dA0_b = dza1.sum(0)  # (64,)
        d_a2_from_adv = dza1 @ self.a1.W.T  # (batch, hidden)

        # --- Value stream backward (v1: hidden→64, v2: 64→1) ---
        # Forward: zv1 = a2_shared @ v1.W + v1.b  shape (batch, 64)
        #          av1 = relu(zv1)
        #          V   = av1 @ v2.W + v2.b          shape (batch, 1)
        # Backward:
        dav1 = dV @ self.v2.W.T  # (batch, 64)
        dzv1 = dav1 * _relu_grad(self._cache["zv1"])  # (batch, 64)
        dV1_W = self._cache["av1"].T @ dV  # (64, 1)
        dV1_b = dV.sum(0).squeeze()  # (1,) → scalar-safe
        dV0_W = self._cache["a2"].T @ dzv1  # (hidden, 64)
        dV0_b = dzv1.sum(0)  # (64,)
        d_a2_from_val = dzv1 @ self.v1.W.T  # (batch, hidden)

        # --- Shared trunk backward ---
        d_a2 = d_a2_from_adv + d_a2_from_val  # (batch, hidden)
        d_z2 = d_a2 * _relu_grad(self._cache["z2"])  # (batch, hidden)
        dfc2_W = self._cache["a1"].T @ d_z2  # (hidden, hidden)
        dfc2_b = d_z2.sum(0)  # (hidden,)
        d_a1 = d_z2 @ self.fc2.W.T  # (batch, hidden)
        d_z1 = d_a1 * _relu_grad(self._cache["z1"])  # (batch, hidden)
        dfc1_W = s.T @ d_z1  # (state_dim, hidden)
        dfc1_b = d_z1.sum(0)  # (hidden,)

        # --- Global gradient-norm clip ---
        all_grads = [
            dfc1_W,
            dfc1_b,
            dfc2_W,
            dfc2_b,
            dV0_W,
            dV0_b,
            dV1_W,
            dV1_b,
            dA0_W,
            dA0_b,
            dA1_W,
            dA1_b,
        ]
        gnorm = np.sqrt(sum(float(np.sum(g**2)) for g in all_grads))
        if gnorm > self.clip_norm:
            scale = self.clip_norm / gnorm
            all_grads = [g * scale for g in all_grads]
        (dfc1_W, dfc1_b, dfc2_W, dfc2_b, dV0_W, dV0_b, dV1_W, dV1_b, dA0_W, dA0_b, dA1_W, dA1_b) = (
            all_grads
        )

        # --- Adam updates ---
        self._t += 1
        kw = dict(lr=self.lr, beta1=self.BETA1, beta2=self.BETA2, eps=self.EPS_ADAM, t=self._t)
        self.fc1.update(dfc1_W, dfc1_b, **kw)
        self.fc2.update(dfc2_W, dfc2_b, **kw)
        self.v1.update(dV0_W, dV0_b, **kw)
        self.v2.update(dV1_W, dV1_b.reshape(-1), **kw)
        self.a1.update(dA0_W, dA0_b, **kw)
        self.a2.update(dA1_W, dA1_b, **kw)

        return loss, td_errs

    # ------------------------------------------------------------------
    # Target-network helpers
    # ------------------------------------------------------------------

    def copy_from(self, other: DuelingQNetwork) -> None:
        """Hard copy all weights (used at init)."""
        for dst, src in self._param_pairs(other):
            dst.W = src.W.copy()
            dst.b = src.b.copy()

    def soft_update_from(self, other: DuelingQNetwork, tau: float) -> None:
        """Polyak averaging: self ← tau*other + (1-tau)*self."""
        for dst, src in self._param_pairs(other):
            dst.W = tau * src.W + (1.0 - tau) * dst.W
            dst.b = tau * src.b + (1.0 - tau) * dst.b

    def _param_pairs(self, other: DuelingQNetwork):
        return zip(
            [self.fc1, self.fc2, self.v1, self.v2, self.a1, self.a2],
            [other.fc1, other.fc2, other.v1, other.v2, other.a1, other.a2],
            strict=True,
        )

In [ ]:
"""
replay_buffer.py  —  Prioritized Experience Replay (PER) + N-step returns
==========================================================================
Algorithm
---------
Schaul et al. (2016) "Prioritized Experience Replay" (arXiv:1511.05952)

Components
----------
1. SumTree        – O(log n) priority-based sampling
2. NStepBuffer    – accumulates n-step returns before pushing to main buffer
3. PrioritizedReplayBuffer – main API used by the agent

How PER works
-------------
Each transition is stored with a priority p_i = |TD_error| + eps.
The sampling probability is  P(i) = p_i^alpha / sum(p_j^alpha).
To correct for the non-uniform sampling bias, we weight each gradient
update by  w_i = (1/N * 1/P(i))^beta,  normalised by max(w_i).
beta is annealed from beta_start → 1.0 over training (full correction at
convergence when the policy is near-optimal).
"""

from __future__ import annotations

from collections import deque

# ---------------------------------------------------------------------------
# SumTree
# ---------------------------------------------------------------------------


class SumTree:
    """
    Binary tree where leaves hold priorities and parents hold sums.

    capacity: number of leaf nodes (rounded up to next power of 2).
    """

    def __init__(self, capacity: int) -> None:
        self.capacity = capacity
        self._tree = np.zeros(2 * capacity, dtype=np.float64)
        self._write = 0  # circular write pointer (leaf index)
        self._n_entries = 0

    # ------------------------------------------------------------------
    # Internal helpers
    # ------------------------------------------------------------------

    def _propagate(self, leaf_idx: int, delta: float) -> None:
        """Propagate a priority change up the tree."""
        idx = leaf_idx
        while idx > 1:
            idx >>= 1
            self._tree[idx] += delta

    def _retrieve(self, idx: int, s: float) -> int:
        """Find the leaf index for query value s (tree traversal)."""
        while True:
            left = 2 * idx
            right = left + 1
            if left >= len(self._tree):
                return idx
            if s <= self._tree[left]:
                idx = left
            else:
                s -= self._tree[left]
                idx = right

    # ------------------------------------------------------------------
    # Public API
    # ------------------------------------------------------------------

    @property
    def total(self) -> float:
        return float(self._tree[1])

    def add(self, priority: float) -> int:
        """
        Insert a new leaf with given priority.
        Returns the leaf index (for use as data-array index).
        """
        leaf_idx = self._write + self.capacity
        self.update(leaf_idx, priority)
        self._write = (self._write + 1) % self.capacity
        self._n_entries = min(self._n_entries + 1, self.capacity)
        return self._write - 1  # data-array index

    def update(self, leaf_idx: int, priority: float) -> None:
        """Update the priority of an existing leaf (leaf_idx is tree index)."""
        delta = priority - self._tree[leaf_idx]
        self._tree[leaf_idx] = priority
        self._propagate(leaf_idx, delta)

    def sample(self, s: float) -> tuple[int, float]:
        """
        Sample one transition by value s ∈ [0, total].
        Returns (data_array_index, priority).
        """
        leaf_idx = self._retrieve(1, s)
        data_idx = leaf_idx - self.capacity
        return data_idx, float(self._tree[leaf_idx])

    def __len__(self) -> int:
        return self._n_entries


# ---------------------------------------------------------------------------
# N-step buffer (accumulates short trajectories)
# ---------------------------------------------------------------------------


class NStepBuffer:
    """
    Collects n consecutive (s, a, r, s', done) tuples and emits a single
    n-step transition:

        G_n = r_0 + gamma*r_1 + ... + gamma^(n-1)*r_{n-1}
        s_n = s_n  (the state n steps ahead)

    When the episode ends before n steps, the buffer is flushed.
    """

    def __init__(self, n: int, gamma: float) -> None:
        self.n = n
        self.gamma = gamma
        self._buf: deque = deque()

    def push(self, s, a, r, s_next, done) -> list[tuple]:
        """
        Add one transition. Returns a list of ready n-step transitions
        (usually 0 or 1 items; all remaining items on done=True).
        """
        self._buf.append((s, a, r, s_next, done))
        ready = []

        if len(self._buf) == self.n:
            ready.append(self._make())
            self._buf.popleft()

        if done:
            # Flush remaining partial transitions
            while self._buf:
                ready.append(self._make())
                self._buf.popleft()

        return ready

    def _make(self) -> tuple:
        """Build one n-step transition from the current buffer."""
        s0, a0, _, _, _ = self._buf[0]
        G = 0.0
        last_s_next = None
        last_done = False
        for i, (_, _, r, s_next, done) in enumerate(self._buf):
            G += (self.gamma**i) * r
            last_s_next = s_next
            last_done = done
            if done:
                break
        return s0, a0, G, last_s_next, last_done


# ---------------------------------------------------------------------------
# Prioritized Replay Buffer
# ---------------------------------------------------------------------------


class PrioritizedReplayBuffer:
    """
    Prioritized Experience Replay buffer.

    Parameters
    ----------
    capacity     : int   – max transitions stored
    alpha        : float – priority exponent (0=uniform, 1=full priority)
    beta_start   : float – initial IS-weight exponent (annealed to 1.0)
    beta_frames  : int   – number of frames over which beta reaches 1.0
    eps_priority : float – small constant to ensure non-zero priority
    n_step       : int   – n-step return accumulation
    gamma        : float – discount factor
    """

    def __init__(
        self,
        capacity: int = 50_000,
        alpha: float = 0.6,
        beta_start: float = 0.4,
        beta_frames: int = 100_000,
        eps_priority: float = 1e-6,
        n_step: int = 3,
        gamma: float = 0.99,
    ) -> None:
        self.capacity = capacity
        self.alpha = alpha
        self.beta_start = beta_start
        self.beta_frames = beta_frames
        self.eps_priority = eps_priority

        self._tree = SumTree(capacity)
        self._data: list = [None] * capacity
        self._max_priority = 1.0
        self._frame = 0

        self._nstep = NStepBuffer(n_step, gamma)

    # ------------------------------------------------------------------
    # Beta annealing
    # ------------------------------------------------------------------

    @property
    def beta(self) -> float:
        frac = min(1.0, self._frame / self.beta_frames)
        return self.beta_start + frac * (1.0 - self.beta_start)

    # ------------------------------------------------------------------
    # Add / Sample
    # ------------------------------------------------------------------

    def push(self, s, a, r, s_next, terminated: bool) -> None:
        """
        Add a raw transition. The n-step buffer will emit ready transitions
        into the SumTree automatically.
        """
        self._frame += 1
        ready = self._nstep.push(s, a, r, s_next, terminated)
        for transition in ready:
            s0, a0, G, sn, dn = transition
            priority = self._max_priority**self.alpha
            idx = self._tree.add(priority)
            self._data[idx] = (
                np.asarray(s0, dtype=np.float32),
                int(a0),
                float(G),
                np.asarray(sn, dtype=np.float32),
                float(dn),
            )

    def sample(self, batch_size: int):
        """
        Draw a prioritized mini-batch.

        Returns
        -------
        states      : (batch, 4)  float32
        actions     : (batch,)    int64
        returns     : (batch,)    float32  – n-step discounted returns
        next_states : (batch,)    float32
        dones       : (batch,)    float32
        weights     : (batch,)    float32  – IS correction weights
        idxs        : list[int]            – leaf indices for priority update
        """
        n = len(self._tree)
        total = self._tree.total
        segment = total / batch_size
        beta = self.beta

        idxs, priorities, samples = [], [], []
        for i in range(batch_size):
            lo, hi = segment * i, segment * (i + 1)
            s_val = np.random.uniform(lo, hi)
            data_idx, priority = self._tree.sample(s_val)
            idxs.append(data_idx + self.capacity)  # tree leaf index
            priorities.append(priority)
            samples.append(self._data[data_idx])

        # IS weights
        min_prob = np.min(priorities) / total
        max_weight = (min_prob * n) ** (-beta)
        probs = np.array(priorities) / total
        weights = ((probs * n) ** (-beta)) / max_weight
        weights = np.asarray(weights, dtype=np.float32)

        s, a, r, s_next, done = zip(*samples, strict=True)
        return (
            np.array(s, dtype=np.float32),
            np.array(a, dtype=np.int64),
            np.array(r, dtype=np.float32),
            np.array(s_next, dtype=np.float32),
            np.array(done, dtype=np.float32),
            weights,
            idxs,
        )

    def update_priorities(self, idxs: list[int], td_errors: np.ndarray) -> None:
        """Update tree priorities given new TD errors."""
        for idx, err in zip(idxs, td_errors, strict=True):
            p = (abs(float(err)) + self.eps_priority) ** self.alpha
            self._max_priority = max(self._max_priority, p)
            self._tree.update(idx, p)

    def __len__(self) -> int:
        return len(self._tree)

    def is_ready(self, batch_size: int) -> bool:
        return len(self) >= batch_size

In [ ]:
"""
dqn_agent.py  —  Rainbow-lite agent for CartPole
=================================================
Techniques implemented
----------------------
1. Double DQN         – online net selects action, target net evaluates it.
                        Eliminates Q-overestimation bias.
2. Dueling network    – separate V(s) and A(s,a) streams (in network.py).
3. Prioritized Replay – PER SumTree with n-step returns (replay_buffer.py).
4. Soft target update – Polyak averaging tau=0.005 every gradient step.
                        Smooth target drift instead of hard-copy jumps.
5. Adam + Huber loss  – stable optimisation even with large TD errors.
6. Gradient clipping  – global norm clip=10 prevents exploding gradients.

Expected convergence: ~200–400 episodes to avg100 >= 475.
"""

from __future__ import annotations

from network import DuelingQNetwork
from replay_buffer import PrioritizedReplayBuffer


class RainbowLiteAgent:
    """
    Parameters
    ----------
    state_dim      : observation dimensionality (4 for CartPole)
    action_dim     : number of discrete actions (2 for CartPole)
    lr             : Adam learning rate
    gamma          : discount factor
    epsilon_start  : initial exploration rate
    epsilon_min    : minimum exploration rate
    epsilon_decay  : multiplicative decay per episode
    batch_size     : PER mini-batch size
    buffer_capacity: max transitions in replay
    tau            : Polyak soft-update coefficient
    n_step         : n-step return length
    alpha          : PER priority exponent
    beta_start     : PER IS-weight initial exponent
    beta_frames    : frames over which beta anneals to 1.0
    hidden         : neurons per hidden layer
    """

    def __init__(
        self,
        state_dim: int = 4,
        action_dim: int = 2,
        lr: float = 5e-4,
        gamma: float = 0.99,
        epsilon_start: float = 1.0,
        epsilon_min: float = 0.001,
        epsilon_decay: float = 0.990,
        batch_size: int = 64,
        buffer_capacity: int = 50_000,
        tau: float = 0.005,
        n_step: int = 3,
        alpha: float = 0.6,
        beta_start: float = 0.4,
        beta_frames: int = 100_000,
        hidden: int = 128,
    ) -> None:
        self.action_dim = action_dim
        self.gamma = gamma
        self.epsilon = epsilon_start
        self.epsilon_min = epsilon_min
        self.epsilon_decay = epsilon_decay
        self.batch_size = batch_size
        self.tau = tau
        self.n_step = n_step

        # Networks
        self.q_online = DuelingQNetwork(state_dim, action_dim, hidden=hidden, lr=lr)
        self.q_target = DuelingQNetwork(state_dim, action_dim, hidden=hidden, lr=lr)
        self.q_target.copy_from(self.q_online)

        # PER buffer (includes n-step accumulator internally)
        self.buffer = PrioritizedReplayBuffer(
            capacity=buffer_capacity,
            alpha=alpha,
            beta_start=beta_start,
            beta_frames=beta_frames,
            n_step=n_step,
            gamma=gamma,
        )

    # ------------------------------------------------------------------
    # Policy
    # ------------------------------------------------------------------

    def select_action(self, state: np.ndarray) -> int:
        """
        Epsilon-greedy action selection.
        Exploration: random action with probability epsilon.
        Exploitation: argmax Q(s, .) from the online network.
        """
        if np.random.rand() < self.epsilon:
            return np.random.randint(self.action_dim)
        q = self.q_online.forward(state[np.newaxis])  # (1, 2)
        return int(np.argmax(q))

    # ------------------------------------------------------------------
    # Experience storage
    # ------------------------------------------------------------------

    def store(
        self,
        state: np.ndarray,
        action: int,
        reward: float,
        next_state: np.ndarray,
        terminated: bool,
    ) -> None:
        """
        Push one raw transition into the n-step / PER buffer.
        Pass `terminated` (NOT `terminated or truncated`) so truncations
        don't incorrectly zero-out the bootstrap value.
        """
        self.buffer.push(state, action, reward, next_state, terminated)

    # ------------------------------------------------------------------
    # Learning step
    # ------------------------------------------------------------------

    def train_step(self) -> float | None:
        """
        One Double-DQN gradient step with PER-weighted Huber loss.

        Returns scalar loss (for logging), or None if buffer not ready.
        """
        if not self.buffer.is_ready(self.batch_size):
            return None

        s, a, r, s_next, done, weights, tree_idxs = self.buffer.sample(self.batch_size)

        # --- Double DQN target ---
        # 1. Online net picks the BEST ACTION in s_next
        q_online_next = self.q_online.forward(s_next)  # (batch, 2)
        best_actions = np.argmax(q_online_next, axis=1)  # (batch,)

        # 2. Target net EVALUATES that action (decoupled → less overestimation)
        q_target_next = self.q_target.forward(s_next)  # (batch, 2)
        q_next_eval = q_target_next[np.arange(self.batch_size), best_actions]

        # Bellman target (n-step discount already baked into r from NStepBuffer)
        targets = r + (self.gamma**self.n_step) * q_next_eval * (1.0 - done)

        # --- Gradient step + get TD errors for PER update ---
        loss, td_errs = self.q_online.backward(s, targets, a, weights)

        # --- Update priorities in the SumTree ---
        self.buffer.update_priorities(tree_idxs, td_errs)

        # --- Soft target update after every gradient step ---
        self.q_target.soft_update_from(self.q_online, tau=self.tau)

        return loss

    # ------------------------------------------------------------------
    # Episode bookkeeping
    # ------------------------------------------------------------------

    def end_episode(self) -> None:
        """Decay epsilon. Call once per completed episode."""
        self.epsilon = max(self.epsilon_min, self.epsilon * self.epsilon_decay)

In [ ]:
"""
train.py  —  Rainbow-lite training loop for CartPole-v1
========================================================
Usage
-----
    uv run python train.py             # headless, saves training_curve.png
    uv run python train.py --render    # show pygame window each episode
    uv run python train.py --episodes 600
"""

from __future__ import annotations

import argparse

import gymnasium as gym
import matplotlib

matplotlib.use("Agg")  # headless backend — works without a display
import matplotlib.pyplot as plt

from dqn_agent import RainbowLiteAgent

# ---------------------------------------------------------------------------
# Config
# ---------------------------------------------------------------------------

NUM_EPISODES = 250
SOLVE_THRESHOLD = 475  # 100-ep rolling average required to declare solved
WARMUP_STEPS = 500  # collect this many transitions before training

# ---------------------------------------------------------------------------
# Helpers
# ---------------------------------------------------------------------------


def make_env(render: bool = False) -> gym.Env:
    return gym.make("CartPole-v1", render_mode="human" if render else None)


def plot_rewards(rewards: list[float], path: str = "training_curve.png") -> None:
    """Save a clean, annotated learning-curve plot."""
    fig, ax = plt.subplots(figsize=(12, 5))
    eps = np.arange(len(rewards))

    ax.plot(eps, rewards, alpha=0.35, color="#74c0fc", linewidth=0.8, label="Episode reward")

    window = 100
    if len(rewards) >= window:
        roll = np.convolve(rewards, np.ones(window) / window, mode="valid")
        ax.plot(
            np.arange(window - 1, len(rewards)),
            roll,
            color="#e03131",
            linewidth=2.2,
            label=f"{window}-ep rolling avg",
        )

        # Mark where the agent solved the task
        solved = np.where(roll >= SOLVE_THRESHOLD)[0]
        if len(solved):
            solve_ep = solved[0] + window - 1
            ax.axvline(solve_ep, color="#2f9e44", linestyle="--", alpha=0.8)
            ax.annotate(
                f"Solved ep {solve_ep}",
                xy=(solve_ep, SOLVE_THRESHOLD),
                xytext=(solve_ep + 10, SOLVE_THRESHOLD - 50),
                arrowprops=dict(arrowstyle="->", color="#2f9e44"),
                color="#2f9e44",
                fontsize=9,
            )

    ax.axhline(
        SOLVE_THRESHOLD,
        color="#2f9e44",
        linestyle=":",
        alpha=0.5,
        label=f"Solve threshold ({SOLVE_THRESHOLD})",
    )
    ax.axhline(500, color="#868e96", linestyle=":", alpha=0.4, label="Max reward (500)")

    ax.set_xlabel("Episode", fontsize=11)
    ax.set_ylabel("Total reward", fontsize=11)
    ax.set_title(
        "Rainbow-lite DQN on CartPole-v1\n"
        "(Double + Dueling + PER + N-step + Soft-target + Adam + Huber)",
        fontsize=12,
    )
    ax.legend(fontsize=9)
    ax.set_ylim(bottom=0)
    fig.tight_layout()
    fig.savefig(path, dpi=160)
    print(f"[PLOT] Saved -> {path}")
    plt.close(fig)


# ---------------------------------------------------------------------------
# Training loop
# ---------------------------------------------------------------------------


def train(num_episodes: int = NUM_EPISODES, render: bool = False) -> list[float]:
    env = make_env(render=render)
    agent = RainbowLiteAgent()

    episode_rewards: list[float] = []
    rolling: deque = deque(maxlen=100)
    solved_at: int | None = None
    total_steps = 0

    header = f"{'Ep':>6} | {'Steps':>7} | {'Reward':>7} | {'Avg100':>7} | {'eps':>6} | {'Loss':>9} | {'beta':>5}"
    sep = "-" * len(header)
    print(header)
    print(sep)

    for ep in range(num_episodes):
        state, _ = env.reset()
        total_reward = 0.0
        last_loss: float | None = None
        done = False

        # ---- inner step loop ----
        while not done:
            action = agent.select_action(state)
            next_state, reward, terminated, truncated, _ = env.step(action)
            done = terminated or truncated
            total_steps += 1

            # Store — use terminated only (not truncated) for Bellman correctness
            agent.store(state, action, reward, next_state, terminated)

            # Warm-up: fill buffer before training starts
            if total_steps >= WARMUP_STEPS:
                loss = agent.train_step()
                if loss is not None:
                    last_loss = loss

            state = next_state
            total_reward += reward

        # ---- episode bookkeeping ----
        agent.end_episode()
        episode_rewards.append(total_reward)
        rolling.append(total_reward)
        avg100 = float(np.mean(rolling))
        beta = agent.buffer.beta

        loss_str = f"{last_loss:9.4f}" if last_loss is not None else "         -"
        print(
            f"{ep:>6} | {total_steps:>7} | {total_reward:>7.1f} |"
            f" {avg100:>7.1f} | {agent.epsilon:>6.3f} | {loss_str} | {beta:>5.3f}"
        )

        # ---- solved? ----
        if avg100 >= SOLVE_THRESHOLD and solved_at is None:
            solved_at = ep
            print(sep)
            print(
                f"[SOLVED] Episode {ep}  |  avg100 = {avg100:.1f}  |  total steps = {total_steps:,}"
            )
            print(sep)
            break

    env.close()

    if solved_at is None:
        print(sep)
        print(
            f"[WARN] Did not solve in {num_episodes} episodes (best avg100 = {max(np.convolve(episode_rewards, np.ones(min(100, len(episode_rewards))) / min(100, len(episode_rewards)), mode='valid')):.1f})"
        )

    return episode_rewards


# ---------------------------------------------------------------------------
# Entry point
# ---------------------------------------------------------------------------

if __name__ == "__main__":
    parser = argparse.ArgumentParser(description="Rainbow-lite DQN on CartPole-v1")
    parser.add_argument("--render", action="store_true", help="Render while training")
    parser.add_argument("--episodes", type=int, default=NUM_EPISODES)
    args = parser.parse_args()

    rewards = train(num_episodes=args.episodes, render=args.render)
    plot_rewards(rewards)